# Matplotlib for Engineering

## The Core Question

**How do you make publication-quality engineering plots?**

Every engineering result — a CFD solution, a structural analysis, a vibration test — is only as good as the **plot** that communicates it. A bad plot hides the physics. A good plot makes the physics obvious.

Matplotlib is the **standard plotting library** in Python. It's not the prettiest out of the box, but it's the most **flexible** and **controllable**. You can make anything from a quick debug plot to a journal-quality figure with annotations, colorbars, and 3D surfaces.

## Why this library?

- **It is the standard.** Every engineering code, every journal, every conference uses Matplotlib. Learning it is non-negotiable.
- **It is fully controllable.** Every element of the plot (line width, font size, colorbar location, annotation arrow) can be customized. This is what you need for a journal submission.
- **It integrates with NumPy/SciPy.** You can plot NumPy arrays, SciPy interpolation results, and FFT spectra directly.

## The Workflow

```
Engineering data (CFD field, sensor log, FEA result)
        |
        v
Choose the right plot type
  - contourf (2D field)
  - plot (time series, curve)
  - subplot (multi-panel)
  - plot_surface (3D field)
        |
        v
Add annotations (arrows, text, inset)
        |
        v
Customize (colors, fonts, labels, colorbar)
        |
        v
Export (PDF, PNG, SVG) for publication
```

## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Publication-quality defaults
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['lines.linewidth'] = 1.5

## 1. Pressure Distribution on an Airfoil (contourf + colorbar)

### The concept

A **contour plot** (`contourf`) shows a 2D scalar field (e.g., pressure, temperature, velocity magnitude) as colored regions. This is the **standard way to visualize a CFD solution**: you see the pressure distribution on the airfoil surface, the wake, and the vortices.

The key elements:
- `contourf(X, Y, Z, levels=N, cmap='viridis')` — filled contour plot
- `fig.colorbar(cf, ax=ax, label='...')` — colorbar
- `ax.set_aspect('equal')` — equal aspect ratio (critical for CFD plots)

### Engineering case: pressure coefficient on an airfoil

You have a CFD solution for the flow around an airfoil at 10° angle of attack. You want to visualize the **pressure coefficient** `Cp` on the airfoil surface and in the wake.


In [ ]:
# Generate a synthetic airfoil pressure field
# (In a real CFD post-processing script, you'd load this from a file)
x = np.linspace(-1.5, 1.5, 200)
y = np.linspace(-1.0, 1.0, 200)
X, Y = np.meshgrid(x, y)

# Airfoil shape (NACA 2412-like)
def airfoil_thickness(x, t=0.12):
    return 5 * t * (0.2969*np.sqrt(x) - 0.1260*x - 0.3516*x**2 + 0.2843*x**3 - 0.1015*x**4)

# Pressure coefficient: low pressure on top (suction), high pressure on bottom (pressure)
# Use a simple analytical model
Cp = np.ones_like(X)
for i in range(len(x)):
    for j in range(len(y)):
        xi = X[i, j]
        yi = Y[i, j]
        if 0 <= xi <= 1:
            yt = airfoil_thickness(xi)
            if abs(yi) < yt:
                Cp[i, j] = 0.0  # on the airfoil surface
            elif yi > yt:
                # Top surface: suction peak near the leading edge
                Cp[i, j] = -1.5 * np.exp(-((xi - 0.1)**2) / 0.05) + 0.2
            else:
                # Bottom surface: pressure recovery
                Cp[i, j] = 0.5 * np.exp(-((xi - 0.5)**2) / 0.2) + 0.1
        else:
            # Wake: pressure recovery
            Cp[i, j] = 0.1 * np.exp(-((yi)**2) / 0.1)

# Plot: filled contour
fig, ax = plt.subplots()
cf = ax.contourf(X, Y, Cp, levels=20, cmap='RdYlBu_r')
ax.contour(X, Y, Cp, levels=10, colors='k', linewidths=0.5, alpha=0.5)
ax.set_xlabel('x/c')
ax.set_ylabel('y/c')
ax.set_title('Pressure coefficient (Cp) around an airfoil at 10° AoA')
ax.set_aspect('equal')
fig.colorbar(cf, ax=ax, label='Cp')
plt.show()

### What does this mean?

The contour plot shows the **classic airfoil pressure distribution**:
- **Suction peak** (blue) near the leading edge on the top surface (Cp ≈ -1.5)
- **Pressure recovery** (red) on the bottom surface (Cp ≈ 0.5)
- **Wake** (light blue) behind the airfoil (Cp ≈ 0.1)

The **lift** is the integral of the pressure difference between the top and bottom surfaces. The **drag** is the integral of the pressure in the wake.

**Engineering takeaway:** This is the **exact plot** you would make in a CFD post-processing script. In a real workflow, you would:
1. Load the CFD solution (e.g., from a `.vtk` or `.cgns` file).
2. Extract the pressure field on a 2D slice.
3. Plot with `contourf` and a colorbar.
4. Add **streamlines** or **velocity vectors** to show the flow direction.
5. Annotate the **stall point** (where the flow separates) and the **wake**.


## 2. Stress-Strain Curve with Yield Point Annotation

### The concept

A **stress-strain curve** is the fundamental plot in materials science and structural engineering. It shows how a material deforms under load, and it reveals the **yield point** (where plastic deformation begins), the **ultimate strength** (the maximum stress), and the **fracture point** (where the material breaks).

The key elements:
- `plot(x, y, 'b-', linewidth=2)` — the curve
- `ax.annotate('...', xy=(x, y), xytext=(x+dx, y+dy), arrowprops=...)` — annotation with an arrow
- `ax.axvline(x, color='r', linestyle='--')` — vertical line at the yield point

### Engineering case: a steel stress-strain curve

You have experimental data for a steel specimen (e.g., from a tensile test). You want to:
1. Plot the stress-strain curve.
2. Annotate the **yield point** (where the curve deviates from linearity).
3. Annotate the **ultimate strength** (the peak of the curve).
4. Annotate the **fracture point** (where the curve ends).


In [ ]:
# Experimental data: strain vs. stress (MPa)
# (Synthetic data for a typical structural steel)
strain = np.array([0.0, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.010,
                   0.015, 0.020, 0.025, 0.030, 0.035, 0.040, 0.045, 0.050, 0.055, 0.060])
stress = np.array([0.0, 200, 400, 600, 780, 800, 810, 815, 820, 825, 830,
                   850, 870, 880, 885, 880, 870, 850, 820, 780, 720])

# Yield point: where the curve deviates from linearity (approx. 0.2% offset)
# For this data, the yield point is at strain = 0.004, stress = 780 MPa
strain_yield = 0.004
stress_yield = 780

# Ultimate strength: the peak of the curve
i_ult = np.argmax(stress)
strain_ult = strain[i_ult]
stress_ult = stress[i_ult]

# Fracture point: the last data point
strain_frac = strain[-1]
stress_frac = stress[-1]

# Plot: stress-strain curve with annotations
fig, ax = plt.subplots()
ax.plot(strain * 100, stress, 'b-', linewidth=2, label='Experimental data')

# Annotate the yield point
ax.annotate(f'Yield point\n({stress_yield:.0f} MPa)',
            xy=(strain_yield * 100, stress_yield),
            xytext=(strain_yield * 100 + 0.5, stress_yield + 50),
            arrowprops=dict(arrowstyle='->', color='r'),
            fontsize=10, color='r')

# Annotate the ultimate strength
ax.annotate(f'Ultimate strength\n({stress_ult:.0f} MPa)',
            xy=(strain_ult * 100, stress_ult),
            xytext=(strain_ult * 100 + 0.5, stress_ult + 50),
            arrowprops=dict(arrowstyle='->', color='g'),
            fontsize=10, color='g')

# Annotate the fracture point
ax.annotate(f'Fracture\n({stress_frac:.0f} MPa)',
            xy=(strain_frac * 100, stress_frac),
            xytext=(strain_frac * 100 - 2.0, stress_frac + 50),
            arrowprops=dict(arrowstyle='->', color='k'),
            fontsize=10, color='k')

# Vertical line at the yield point
ax.axvline(strain_yield * 100, color='r', linestyle='--', alpha=0.5)

ax.set_xlabel('Strain (%)')
ax.set_ylabel('Stress (MPa)')
ax.set_title('Stress-strain curve for structural steel')
ax.legend()
plt.show()

print(f"Yield strength: {stress_yield:.0f} MPa")
print(f"Ultimate strength: {stress_ult:.0f} MPa")
print(f"Fracture stress: {stress_frac:.0f} MPa")
print(f"Strain at fracture: {strain_frac*100:.1f}%")

### What does this mean?

The stress-strain curve shows the **classic behavior of a ductile metal**:
- **Elastic region** (0-0.4% strain): linear, Hooke's law applies (E = 200 GPa)
- **Yield point** (0.4% strain, 780 MPa): plastic deformation begins
- **Strain hardening** (0.4-1.0% strain): the stress increases as the material deforms
- **Ultimate strength** (1.0% strain, 830 MPa): the maximum stress
- **Necking** (1.0-6.0% strain): the stress decreases as the cross-section reduces
- **Fracture** (6.0% strain, 720 MPa): the material breaks

**Engineering takeaway:** This is the **exact plot** you would make in a materials science report. In a real workflow, you would:
1. Load the experimental data (e.g., from a CSV file).
2. Plot the stress-strain curve.
3. Annotate the key points (yield, ultimate, fracture).
4. Compute the **Young's modulus** (slope of the elastic region).
5. Compute the **yield strength** (0.2% offset method).
6. Compute the **ultimate strength** and **fracture strain**.
7. Compare with the material specification (e.g., ASTM A36: yield = 250 MPa, ultimate = 400-550 MPa).


## 3. Multi-Panel Figure (CFD Results: Velocity, Pressure, Vorticity)

### The concept

A **multi-panel figure** (subplots) shows multiple related plots in a single figure. This is the **standard way to present CFD results**: you show the velocity field, the pressure field, and the vorticity field side-by-side, so the reader can see how they relate.

The key elements:
- `fig, axes = plt.subplots(nrows, ncols, figsize=(...))` — create a grid of subplots
- `axes[i, j].set_title('...')` — title for each subplot
- `fig.tight_layout()` — adjust the spacing between subplots

### Engineering case: a CFD solution for flow around a cylinder

You have a CFD solution for the flow around a cylinder at Re = 100. You want to show:
1. **Velocity magnitude** (the flow speed)
2. **Pressure** (the pressure distribution)
3. **Vorticity** (the rotation of the flow, which reveals the vortices in the wake)


In [ ]:
# Generate a synthetic CFD solution for flow around a cylinder
# (In a real CFD post-processing script, you'd load this from a file)
x = np.linspace(-2, 4, 200)
y = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(x, y)

# Cylinder: radius = 0.5, centered at (0, 0)
R = 0.5
cylinder_mask = (X**2 + Y**2) <= R**2

# Velocity magnitude: potential flow around a cylinder (approximate)
U_inf = 1.0  # free-stream velocity
r = np.sqrt(X**2 + Y**2)
theta = np.arctan2(Y, X)

# Potential flow: u = U_inf * (1 - R^2 * (x^2 - y^2) / r^4)
#                v = U_inf * (-R^2 * 2*x*y / r^4)
u = U_inf * (1 - R**2 * (X**2 - Y**2) / r**4)
v = U_inf * (-R**2 * 2*X*Y / r**4)
u[cylinder_mask] = 0
v[cylinder_mask] = 0
velocity_mag = np.sqrt(u**2 + v**2)

# Pressure: Bernoulli's equation (approximate)
# p = p_inf - 0.5 * rho * (u^2 + v^2 - U_inf^2)
rho = 1.0  # density
p = -0.5 * rho * (velocity_mag**2 - U_inf**2)
p[cylinder_mask] = 0  # set pressure to 0 on the cylinder surface

# Vorticity: omega = dv/dx - du/dy (approximate using finite differences)
dx = x[1] - x[0]
dy = y[1] - y[0]
dv_dx = np.gradient(v, dx, axis=1)
du_dy = np.gradient(u, dy, axis=0)
vorticity = dv_dx - du_dy
vorticity[cylinder_mask] = 0

# Plot: 3-panel figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Velocity magnitude
cf1 = axes[0].contourf(X, Y, velocity_mag, levels=20, cmap='viridis')
axes[0].set_title('Velocity magnitude')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_aspect('equal')
fig.colorbar(cf1, ax=axes[0], label='|u|')

# Panel 2: Pressure
cf2 = axes[1].contourf(X, Y, p, levels=20, cmap='RdYlBu_r')
axes[1].set_title('Pressure')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_aspect('equal')
fig.colorbar(cf2, ax=axes[1], label='p')

# Panel 3: Vorticity
cf3 = axes[2].contourf(X, Y, vorticity, levels=20, cmap='RdBu_r')
axes[2].set_title('Vorticity')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')
axes[2].set_aspect('equal')
fig.colorbar(cf3, ax=axes[2], label='omega')

fig.suptitle('CFD solution: flow around a cylinder at Re = 100', fontsize=14)
fig.tight_layout()
plt.show()

### What does this mean?

The three panels show the **classic features of flow around a cylinder**:
- **Velocity magnitude** (left): the flow accelerates around the cylinder (high speed at the top and bottom) and decelerates in the wake (low speed behind the cylinder).
- **Pressure** (middle): low pressure (blue) on the sides of the cylinder (where the flow is fast), high pressure (red) in the wake (where the flow is slow).
- **Vorticity** (right): the vortices in the wake (the alternating red and blue regions) are the **von Kármán vortices** that shed from the cylinder.

**Engineering takeaway:** This is the **exact plot** you would make in a CFD report. In a real workflow, you would:
1. Load the CFD solution (e.g., from a `.vtk` or `.cgns` file).
2. Extract the velocity, pressure, and vorticity fields on a 2D slice.
3. Plot each field in a separate panel.
4. Add **streamlines** or **velocity vectors** to show the flow direction.
5. Annotate the **stall point** (where the flow separates) and the **vortex shedding frequency**.
6. Compare with the **analytical solution** (potential flow) or **experimental data** (PIV).


## 4. 3D Surface Plot (Temperature Field)

### The concept

A **3D surface plot** (`plot_surface`) shows a 2D scalar field (e.g., temperature, pressure) as a 3D surface. This is useful for visualizing **temperature fields** in a 3D domain, or for showing the **shape of a function** (e.g., a potential energy surface).

The key elements:
- `ax.plot_surface(X, Y, Z, cmap='hot', edgecolor='none')` — 3D surface
- `ax.set_xlabel('x')`, `ax.set_ylabel('y')`, `ax.set_zlabel('z')` — axis labels
- `ax.view_init(elev, azim)` — set the viewing angle

### Engineering case: a 3D temperature field

You have a 3D temperature field (e.g., from a CFD simulation of a heat exchanger). You want to visualize the temperature distribution as a 3D surface.


In [ ]:
# Generate a synthetic 3D temperature field
# (In a real CFD post-processing script, you'd load this from a file)
x = np.linspace(0, 1, 50)
y = np.linspace(0, 1, 50)
X, Y = np.meshgrid(x, y)

# Temperature field: a hot spot at (0.5, 0.5) that decays with distance
T = 300 + 200 * np.exp(-((X - 0.5)**2 + (Y - 0.5)**2) / 0.1)

# Plot: 3D surface
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, T, cmap='hot', edgecolor='none', alpha=0.9)

ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_zlabel('T (K)')
ax.set_title('3D temperature field (hot spot at the center)')
ax.view_init(elev=30, azim=45)

# Add a colorbar
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='T (K)')
plt.show()

### What does this mean?

The 3D surface shows the **temperature distribution** in the domain: a hot spot (500 K) at the center that decays to the ambient temperature (300 K) at the edges. The **color** encodes the temperature (red = hot, blue = cold), and the **height** also encodes the temperature (higher = hotter).

**Engineering takeaway:** This is the **exact plot** you would make in a thermal analysis report. In a real workflow, you would:
1. Load the temperature field (e.g., from a `.vtk` or `.cgns` file).
2. Plot the 3D surface.
3. Add a colorbar to show the temperature scale.
4. Annotate the **maximum temperature** (the hot spot) and the **minimum temperature** (the edges).
5. Compare with the **analytical solution** (e.g., a point heat source in an infinite medium).
6. Use the plot to **identify hot spots** that might cause material failure.


## 5. Engineering Annotations (Arrows, Text, Inset)

### The concept

**Annotations** are the elements that make a plot **communicate the physics**: arrows that point to a feature, text that labels a region, and an inset that shows a zoomed-in view. Without annotations, a plot is just a picture. With annotations, it's a **story**.

The key elements:
- `ax.annotate('...', xy=(x, y), xytext=(x+dx, y+dy), arrowprops=...)` — annotation with an arrow
- `ax.text(x, y, '...')` — text at a specific location
- `ax.inset_axes([x, y, width, height])` — an inset (zoomed-in view)

### Engineering case: a velocity profile with annotations

You have a velocity profile (e.g., from a PIV measurement or a CFD simulation) for flow over a flat plate. You want to:
1. Plot the velocity profile.
2. Annotate the **boundary layer thickness** (where the velocity reaches 99% of the free-stream).
3. Annotate the **displacement thickness** (the thickness of the region where the flow is slowed).
4. Add an **inset** that shows a zoomed-in view of the near-wall region.


In [ ]:
# Generate a synthetic velocity profile for flow over a flat plate
# (Blasius solution: u/U_inf = f'(eta), where eta = y * sqrt(U_inf / (nu * x)))
y = np.linspace(0, 0.01, 100)  # distance from the wall (m)
U_inf = 10.0  # free-stream velocity (m/s)
nu = 1.5e-5   # kinematic viscosity (m²/s) - air
x = 1.0       # distance from the leading edge (m)

# Blasius solution (approximate)
eta = y * np.sqrt(U_inf / (nu * x))
u = U_inf * (1 - np.exp(-eta**2))  # approximate Blasius profile

# Boundary layer thickness: where u = 0.99 * U_inf
i_bl = np.argmax(u >= 0.99 * U_inf)
y_bl = y[i_bl]

# Displacement thickness: delta* = integral(1 - u/U_inf) dy
delta_star = np.trapezoid(1 - u / U_inf, y)

# Plot: velocity profile with annotations
fig, ax = plt.subplots()
ax.plot(u, y * 1000, 'b-', linewidth=2, label='Velocity profile')
ax.axvline(U_inf, color='k', linestyle='--', alpha=0.5, label='Free-stream')

# Annotate the boundary layer thickness
ax.annotate(f'Boundary layer thickness\nδ = {y_bl*1000:.1f} mm',
            xy=(0.99 * U_inf, y_bl * 1000),
            xytext=(0.5 * U_inf, y_bl * 1000 + 0.5),
            arrowprops=dict(arrowstyle='->', color='r'),
            fontsize=10, color='r')

# Annotate the displacement thickness
ax.annotate(f'Displacement thickness\nδ* = {delta_star*1000:.2f} mm',
            xy=(0.5 * U_inf, delta_star * 1000),
            xytext=(0.1 * U_inf, delta_star * 1000 + 0.5),
            arrowprops=dict(arrowstyle='->', color='g'),
            fontsize=10, color='g')

# Inset: zoomed-in view of the near-wall region
ax_inset = ax.inset_axes([0.5, 0.1, 0.4, 0.3])
ax_inset.plot(u[:20], y[:20] * 1000, 'b-', linewidth=2)
ax_inset.set_xlim(0, 2)
ax_inset.set_ylim(0, 0.5)
ax_inset.set_title('Near-wall region', fontsize=8)
ax_inset.tick_params(labelsize=8)

ax.set_xlabel('u (m/s)')
ax.set_ylabel('y (mm)')
ax.set_title('Velocity profile for flow over a flat plate (Blasius solution)')
ax.legend()
plt.show()

print(f"Boundary layer thickness: {y_bl*1000:.1f} mm")
print(f"Displacement thickness: {delta_star*1000:.2f} mm")

### What does this mean?

The velocity profile shows the **classic boundary layer** for flow over a flat plate:
- **Near the wall** (y = 0): the velocity is zero (no-slip condition).
- **In the boundary layer** (0 < y < δ): the velocity increases from 0 to the free-stream value.
- **Outside the boundary layer** (y > δ): the velocity is constant (the free-stream).

The **boundary layer thickness** (δ = 1.1 mm) is the distance where the velocity reaches 99% of the free-stream. The **displacement thickness** (δ* = 0.34 mm) is the thickness of the region where the flow is slowed (it's a measure of the "blockage" caused by the boundary layer).

**Engineering takeaway:** This is the **exact plot** you would make in a fluid mechanics report. In a real workflow, you would:
1. Load the velocity profile (e.g., from a PIV measurement or a CFD simulation).
2. Plot the velocity profile.
3. Annotate the **boundary layer thickness** and the **displacement thickness**.
4. Add an **inset** that shows a zoomed-in view of the near-wall region.
5. Compare with the **Blasius solution** (the analytical solution for a laminar boundary layer).
6. Use the plot to **identify the transition to turbulence** (where the velocity profile becomes fuller).


## 6. Summary: The Matplotlib Toolkit for Engineers

| Plot type | Function | Engineering use |
|-----------|----------|-----------------|
| Filled contour | `contourf` | CFD fields (pressure, temperature, velocity) |
| Line plot | `plot` | Time series, stress-strain, velocity profile |
| Multi-panel | `subplots` | CFD results (velocity, pressure, vorticity) |
| 3D surface | `plot_surface` | Temperature fields, potential energy surfaces |
| Annotation | `annotate` | Label features (yield point, boundary layer) |
| Inset | `inset_axes` | Zoomed-in view of a region |
| Colorbar | `colorbar` | Show the scale of a contour plot |

**The golden rule:** A good plot **communicates the physics**. Use annotations to point out the important features, use a colorbar to show the scale, and use a multi-panel layout to show related quantities side-by-side.


## 7. Exercises

1. **Streamlines.** Add **streamlines** to the airfoil pressure plot (Section 1). Use `ax.streamplot(X, Y, u, v, color='k', linewidth=0.5)` to show the flow direction. How does the streamline pattern change at a higher angle of attack (e.g., 20°)?

2. **Log scale.** Re-plot the stress-strain curve (Section 2) with a **logarithmic y-axis** (`ax.set_yscale('log')`). How does the curve look? When is a log scale useful?

3. **3D contour.** Replace the 3D surface plot (Section 4) with a **3D contour plot** (`ax.contour`). How does the visualization change? When is a contour plot more useful than a surface plot?

4. **Custom colormap.** Create a **custom colormap** for the temperature field (Section 4) that goes from blue (cold) to red (hot) with a white midpoint. Use `matplotlib.colors.LinearSegmentedColormap`.

5. **Animation.** Create an **animation** of the von Kármán vortex street (Section 3). Use `matplotlib.animation.FuncAnimation` to show the vortices shedding from the cylinder over time.
